[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabloriospena/lucirme-ai-web/blob/main/public/estudia-ia-basico/clase-3/notebook/Clase3_Basico_Mi_Proyecto_EstudIA.ipynb)

# Estud-IA · IA nivel básico · Clase 3
## Plantilla de tu proyecto: conecta, explora y documenta tu fuente

**Tiempo:** bloque de construcción (60 minutos). **Trabajo:** individual. **Funciona con cualquier fuente tabular**: una hoja de Google publicada como CSV, un archivo que subas, una API pública o una muestra sintética.

### Lo que vas a entregar al terminar
1. Un **diccionario de datos** de tu fuente (`diccionario_proyecto.csv`).
2. **Tres conteos comprobados** contra la fuente.
3. Un **registro de la consulta** que permite repetirla (`registro_proyecto.json` y `.csv`).
4. Una **prueba en frío** hecha con ese registro.

### Cómo se usa
- Ejecuta las celdas **en orden**. Cada parte explica qué hace, qué esperar y qué cuidar.
- Las celdas **TU TURNO** son las únicas que debes editar. El resto se ejecuta tal cual.
- No necesitas IA para nada de esto. Si usas una herramienta con IA para preguntar sobre tus datos, la Parte 8 te prepara una muestra pequeña y segura. Si se acaban tus tokens, el notebook sigue funcionando.

### Las tres reglas de la clase
- **Lo mínimo.** Conecta solo la hoja o la tabla que necesitas.
- **Solo lectura.** Hoy leemos, no escribimos ni borramos en la fuente.
- **Sin datos personales.** Fuentes públicas, sintéticas o con permiso.

### Mapa
| Parte | Qué haces |
|---|---|
| 0 | Preparar el entorno |
| 1 | Elegir y cargar tu fuente; anotar origen y momento |
| 2 | Explorar: tamaño, tipos, vacíos, repetidos y perfil de cada columna |
| 3 | Revisar que no haya datos personales |
| 4 | Diccionario de datos |
| 5 | Tres conteos comprobados |
| 6 | Una pregunta que tu fuente permite y otra que no |
| 7 | Registro y archivos para entregar |
| 8 | Prueba en frío |
| 9 | Una muestra segura para preguntar a una IA |
| 10 | Si algo falla, glosario y lista final |

## Parte 0 · Preparar el entorno

**Qué hace la celda:** importa las bibliotecas y anota las versiones. `pandas` trabaja con tablas; `json` guarda el registro; `io`, `sys` y `datetime` vienen con Python.

**Qué esperar:** dos líneas con las versiones. No pasa nada más.

In [ ]:
import io
import json
import random
import sys
from datetime import datetime, timedelta

import pandas as pd

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

## Parte 1 · Elegir y cargar tu fuente

### Las cuatro opciones
Elige **una** y escríbela en `OPCION` en la celda de configuración.

| `OPCION` | Cuándo usarla | Qué necesitas | Tipo de fuente |
|---|---|---|---|
| `"url"` | Tu fuente es una hoja de Google o un CSV publicado | El enlace CSV | **Viva** |
| `"api"` | Tu fuente es una API pública que devuelve JSON | La dirección de la API | **Viva** |
| `"subir"` | Tienes un archivo CSV o Excel | Subirlo a Colab | **Copia** |
| `"muestra"` | No tienes datos todavía | Nada | **Copia** (inventada) |

**Viva o copia:** con una fuente viva el resultado puede cambiar mañana. Con una copia no, y por eso hay que decir que es una copia y de qué fecha.

### Cómo obtener el enlace CSV de una hoja de Google (`"url"`)
1. Abre tu hoja. Menú **Archivo → Compartir → Publicar en la Web**.
2. Elige la pestaña de datos y el formato **Valores separados por comas (.csv)**. Pulsa **Publicar**.
3. Copia el enlace y pégalo en `URL`.

Un enlace publicado así es de solo lectura. Piensa antes de publicar: **cualquiera con el enlace puede leerlo**. Publica solo hojas que no tengan datos personales ni de clientes.

### Cómo usar una API pública (`"api"`)
Una API devuelve datos por internet. Muchos portales de datos abiertos ofrecen una dirección que responde en formato JSON o CSV. Si la dirección devuelve una lista de registros, este notebook la convierte en tabla con `pd.json_normalize`. Si devuelve un objeto con la lista adentro, escribe en `CLAVE_LISTA` el nombre de esa clave.

### Si eliges `"subir"`
Al ejecutar la celda de carga, Colab abre un botón para elegir tu archivo. Funciona con `.csv` y `.xlsx`. Si no estás en Colab, escribe la ruta del archivo en `ARCHIVO_LOCAL`.

### TU TURNO · configura tu fuente

**Qué hace la celda:** solo guarda tus decisiones en variables. No carga nada todavía.

**Qué editar:** `OPCION` y, según la opción, `URL`, `CLAVE_LISTA` o `ARCHIVO_LOCAL`. Déjalas entre comillas.

In [ ]:
# TU TURNO: elige una opción -> "url", "api", "subir" o "muestra"
OPCION = "muestra"

# Para "url" y "api": pega aquí el enlace o la dirección.
URL = ""

# Solo para "api": si la respuesta es un objeto con la lista adentro, escribe la clave. Si no, déjala vacía.
CLAVE_LISTA = ""

# Solo para "subir" fuera de Colab: ruta del archivo en tu computador.
ARCHIVO_LOCAL = ""

### Las funciones de carga

**Qué hace la celda:** define cuatro funciones, una por opción, y una que decide cuál usar.

- `leer_url`: usa `pd.read_csv`, que sabe leer un CSV desde un enlace.
- `leer_api`: pide la dirección con la biblioteca `urllib` (que viene con Python), convierte el JSON y lo vuelve tabla.
- `leer_archivo`: abre el selector de archivos de Colab, o lee la ruta si no estás en Colab.
- `muestra_sintetica`: crea una tabla inventada de 80 filas (78 pedidos de una tienda más dos filas repetidas a propósito), con vacíos para que haya algo que descubrir. Siempre da lo mismo, porque usa una semilla fija (`random.Random(11)`).

**Qué esperar:** nada visible. Si ves un error aquí, casi siempre es de escritura; revisa las comillas.

**Por qué cada función devuelve tres cosas:** la tabla, un texto de origen y el tipo (`viva` o `copia`). El texto de origen y el tipo van al registro.

In [ ]:
def leer_url(url):
    tabla = pd.read_csv(url)
    return tabla, f"Fuente viva (CSV): {url}", "viva"


def leer_api(url, clave_lista=""):
    from urllib.request import urlopen
    with urlopen(url, timeout=30) as respuesta:
        datos = json.loads(respuesta.read().decode("utf-8"))
    if clave_lista:
        datos = datos[clave_lista]
    if isinstance(datos, dict):
        raise ValueError(
            "La API devolvió un objeto, no una lista. Estas son sus claves: "
            f"{list(datos.keys())}. Escribe en CLAVE_LISTA la que contiene la lista."
        )
    tabla = pd.json_normalize(datos)
    return tabla, f"Fuente viva (API): {url}", "viva"


def leer_archivo(ruta_local=""):
    ruta = ruta_local
    if not ruta:
        try:
            from google.colab import files
            subidos = files.upload()
            ruta = list(subidos.keys())[0]
        except ImportError:
            raise RuntimeError("No estoy en Colab. Escribe la ruta en ARCHIVO_LOCAL.")
    if ruta.lower().endswith((".xlsx", ".xls")):
        tabla = pd.read_excel(ruta)
    else:
        tabla = pd.read_csv(ruta)
    return tabla, f"Copia: archivo '{ruta}'", "copia"


def muestra_sintetica():
    r = random.Random(11)
    productos = {"Café": 18000, "Panela": 6000, "Arepas": 9000, "Chocolate": 12000, "Aceite": 22000}
    inicio = datetime(2026, 9, 1)
    filas = []
    for n in range(1, 79):
        producto = r.choice(list(productos))
        filas.append({
            "pedido": f"P{n:03d}",
            "fecha": (inicio + timedelta(days=r.randrange(0, 50))).strftime("%d/%m/%Y"),
            "ciudad": r.choice(["Medellín", "Bogotá", "Cali", "Barranquilla"]),
            "producto": producto,
            "cantidad": r.randrange(1, 8),
            "valor_unitario": productos[producto],
            "estado": r.choice(["Entregado", "Entregado", "Pendiente", "Cancelado"]),
        })
    tabla = pd.DataFrame(filas)
    for i in r.sample(range(len(tabla)), 4):
        tabla.loc[i, "ciudad"] = None
    for i in r.sample(range(len(tabla)), 6):
        tabla.loc[i, "estado"] = None
    tabla = pd.concat([tabla, tabla.iloc[[5, 40]]], ignore_index=True)   # dos filas repetidas
    return tabla, "Copia: muestra sintética del notebook (datos inventados)", "copia"


def cargar_fuente():
    if OPCION == "url":
        return leer_url(URL)
    if OPCION == "api":
        return leer_api(URL, CLAVE_LISTA)
    if OPCION == "subir":
        return leer_archivo(ARCHIVO_LOCAL)
    if OPCION == "muestra":
        return muestra_sintetica()
    raise ValueError('OPCION debe ser "url", "api", "subir" o "muestra".')

### Cargar y anotar de dónde viene

**Qué hace la celda:** carga tu fuente y guarda la fecha y la hora de la consulta. Muestra los tres datos.

**Qué esperar:** el origen, el tipo y el momento. Ejecútala otra vez: el momento cambia. Si tu fuente es viva y alguien agrega filas, los conteos de más abajo pueden cambiar también. Por eso se anota.

**Si falla,** el error dice la causa. Los más comunes están en la Parte 10.

In [ ]:
df, ORIGEN, TIPO_FUENTE = cargar_fuente()
MOMENTO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Origen :", ORIGEN)
print("Tipo   :", TIPO_FUENTE)
print("Momento:", MOMENTO)
print("Tamaño :", df.shape[0], "filas x", df.shape[1], "columnas")

## Parte 2 · Explorar antes de preguntar

Cuatro vistas rápidas, y después un **perfil de cada columna** que junta lo más útil en una sola tabla.

**Predice antes de ejecutar:** ¿qué representa una fila de tu fuente? ¿Cuántas columnas crees que tiene? ¿En cuál crees que hay vacíos?

In [ ]:
df.head()

**Qué hizo `head()`:** mostró las primeras 5 filas. Lee los nombres de las columnas y busca `NaN`: es como Pandas muestra una **celda vacía**.

In [ ]:
print("Columnas:", list(df.columns))
print()
print(df.dtypes)

**Cómo leer `dtypes`:** `int64` es entero, `float64` decimal, `object` o `str` es texto (según la versión de Pandas verás una u otra). Si una columna que debería ser número aparece como texto, o una fecha aparece como texto, anótalo: lo corregimos en la próxima clase, no hoy.

### El perfil de columnas

**Qué hace la función `perfil_columnas`:** para cada columna calcula
- su **tipo**;
- cuántas celdas están **vacías** y qué porcentaje es;
- cuántas celdas tienen **solo espacios** (parecen vacías pero Pandas no las cuenta como vacías; es el error de conteo más común);
- cuántos **valores distintos** tiene;
- tres **ejemplos**.

**Cómo leerla:**
- Una columna con casi tantos valores distintos como filas suele ser un identificador.
- Una columna con pocos valores distintos (tres o cuatro) suele ser una categoría. Es una buena candidata para tu conteo de categoría.
- Un porcentaje de vacíos alto en una columna que necesitas es una advertencia para tu pregunta.

In [ ]:
def perfil_columnas(tabla):
    filas = []
    for columna in tabla.columns:
        s = tabla[columna]
        solo_espacios = 0
        if s.dtype == object or str(s.dtype) in ("str", "string"):
            solo_espacios = int((s.dropna().astype(str).str.strip() == "").sum())
        ejemplos = s.dropna().astype(str).unique()[:3]
        filas.append({
            "columna": columna,
            "tipo": str(s.dtype),
            "vacíos": int(s.isna().sum()),
            "% vacíos": round(100 * float(s.isna().mean()), 1),
            "solo espacios": solo_espacios,
            "valores distintos": int(s.nunique()),
            "ejemplos": ", ".join(ejemplos),
        })
    return pd.DataFrame(filas)


perfil = perfil_columnas(df)
pd.set_option("display.max_colwidth", 60)
perfil

### Filas repetidas

**Qué hace la celda:** cuenta las filas exactamente repetidas con `duplicated()`.

**Importante:** hoy **no las borres**. Hoy las encuentras y las anotas. La próxima clase decidimos qué hacer con ellas. Una fila repetida puede ser un error, pero también puede ser un registro legítimo (dos pedidos idénticos en el mismo día). Solo tú, que conoces la fuente, puedes decirlo.

In [ ]:
repetidas = int(df.duplicated().sum())
print("Filas exactamente repetidas:", repetidas)
if repetidas:
    print()
    print("Primeras repetidas (la segunda aparición de cada una):")
    display(df[df.duplicated(keep='first')].head())

## Parte 3 · Revisar que no haya datos personales

La regla de la clase es **sin datos personales**. El código no puede garantizarlo, pero puede darte una alerta sencilla: revisa si algún **nombre de columna** parece de un dato personal.

**Qué hace la celda:** busca en los nombres de columna palabras como *nombre*, *cédula*, *teléfono*, *correo*, *dirección* o *documento*. Es una alerta, no una prueba: una columna llamada `x1` podría tener un teléfono y esta revisión no lo vería. **Mira tus datos tú también.**

**Qué hacer si aparece una alerta:** no la ignores. Pregunta si tienes permiso de usar esa columna. Si no, quítala del análisis o usa la muestra sintética.

In [ ]:
PALABRAS_PERSONALES = ["nombre", "apellido", "cedula", "cédula", "documento", "telefono", "teléfono",
                       "celular", "correo", "email", "mail", "direccion", "dirección", "nacimiento"]

alertas = [c for c in df.columns if any(p in str(c).lower() for p in PALABRAS_PERSONALES)]

if alertas:
    print("Alerta: estas columnas parecen datos personales ->", alertas)
    print("Confirma que tienes permiso o quítalas antes de seguir.")
else:
    print("Ningún nombre de columna parece un dato personal.")
    print("Aun así, mira tus datos con tus ojos: una columna con otro nombre puede tenerlos.")

## Parte 4 · Diccionario de datos

Un **diccionario de datos** explica tu fuente a otra persona. Para **cada columna** responde tres cosas:

| Campo | Pregunta | Ejemplo |
|---|---|---|
| `significado` | ¿Qué es esta columna? | «Día en que llegó la solicitud» |
| `formato` | ¿En qué forma viene? | «Texto día/mes/año» |
| `vacio_significa` | ¿Qué significa que esté vacía? | «No lo informó» |

Y una regla de oro: **un vacío no siempre significa lo mismo**, y un `0` no siempre es un error. Tú decides qué significan; por eso el diccionario lo escribes tú y no el código.

### Paso 1 · Genera el esqueleto
**Qué hace la celda:** imprime una plantilla con **tus** columnas, lista para copiar.

### Paso 2 · Pégalo y complétalo
Copia lo que imprimió en la celda siguiente, dentro de las llaves de `DICCIONARIO`, y completa cada campo. Para la muestra sintética ya hay un ejemplo completo que puedes mirar.

In [ ]:
print("Copia desde aquí:\n")
for columna in df.columns:
    print(f'    "{columna}": {{"significado": "", "formato": "", "vacio_significa": ""}},')

### TU TURNO · completa tu diccionario

**Qué hace la celda:** guarda el diccionario en una variable.

Lo que ves es el ejemplo de la **muestra sintética**. Si usas otra fuente, **reemplázalo** con el esqueleto que copiaste y complétalo.

**Un buen significado:** dice qué es, no cómo se llama («Ciudad donde se entregó el pedido», no «la columna ciudad»).
**Un buen `vacio_significa`:** dice una causa posible, o «No hay vacíos» si no los hay.

In [ ]:
DICCIONARIO = {
    "pedido":         {"significado": "Código del pedido", "formato": "Texto: P y tres dígitos", "vacio_significa": "No hay vacíos"},
    "fecha":          {"significado": "Día en que se hizo el pedido", "formato": "Texto día/mes/año", "vacio_significa": "No hay vacíos"},
    "ciudad":         {"significado": "Ciudad a donde se envía el pedido", "formato": "Texto", "vacio_significa": "No se registró la ciudad"},
    "producto":       {"significado": "Producto pedido", "formato": "Texto", "vacio_significa": "No hay vacíos"},
    "cantidad":       {"significado": "Unidades pedidas", "formato": "Número entero", "vacio_significa": "No hay vacíos"},
    "valor_unitario": {"significado": "Precio de una unidad, en pesos", "formato": "Número entero", "vacio_significa": "No hay vacíos"},
    "estado":         {"significado": "Situación del pedido", "formato": "Texto: Entregado, Pendiente o Cancelado", "vacio_significa": "Aún sin estado asignado"},
}

### Revisar el diccionario

**Qué hace la función `revisar_diccionario`:** compara tu diccionario con las columnas reales de la tabla y te dice:
- si **falta** alguna columna en el diccionario;
- si el diccionario tiene columnas que **ya no existen** en la tabla;
- si algún campo quedó **sin completar**.

Después arma una tabla y la muestra. **Cuando no haya avisos, tu diccionario está completo.**

In [ ]:
def revisar_diccionario(diccionario, tabla):
    columnas = list(tabla.columns)
    faltan = [c for c in columnas if c not in diccionario]
    sobran = [c for c in diccionario if c not in columnas]
    sin_completar = [
        f"{c} ({campo})"
        for c, campos in diccionario.items()
        for campo in ("significado", "formato", "vacio_significa")
        if not str(campos.get(campo, "")).strip()
    ]
    if faltan:
        print("Faltan en el diccionario:", faltan)
    if sobran:
        print("Sobran en el diccionario (no están en la tabla):", sobran)
    if sin_completar:
        print("Campos sin completar:", sin_completar)
    if not (faltan or sobran or sin_completar):
        print("Diccionario completo: todas las columnas tienen sus tres campos.")
    filas = [
        [c, d.get("significado", ""), d.get("formato", ""), d.get("vacio_significa", "")]
        for c, d in diccionario.items()
    ]
    return pd.DataFrame(filas, columns=["columna", "significado", "formato", "qué significa un vacío"])


tabla_diccionario = revisar_diccionario(DICCIONARIO, df)
tabla_diccionario

## Parte 5 · Tres conteos comprobados

La regla: **cuentas tú en la fuente, cuenta el código, y se comparan.**

### Los tres tipos de conteo que vamos a usar
| `tipo` | Qué cuenta | Campos que necesita |
|---|---|---|
| `"filas"` | Todas las filas | — |
| `"vacios"` | Celdas vacías | `columna` (opcional; sin ella, cuenta todas) |
| `"categoria"` | Filas donde una columna vale exactamente un valor | `columna` y `valor` |

Hay dos más disponibles: `"distintos"` (cuántos valores distintos tiene `columna`) y `"suma"` (suma de una columna numérica).

### Por qué se escriben como una lista de especificaciones
Cada conteo es una pequeña **ficha**: qué se cuenta, en qué columna y qué valor tú encontraste a mano. Así el registro puede guardar *exactamente qué contaste* y la prueba en frío puede repetirlo sin que recuerdes nada.

### Cómo contar a mano en tu fuente
- **Filas:** última fila menos 1 (la primera es el encabezado).
- **Vacíos de una columna:** filas totales menos las celdas con valor (mira el contador abajo a la derecha al seleccionar la columna).
- **Categoría:** pon un filtro en la columna y cuenta cuántas filas quedan. **Quita el filtro al terminar.**

### Dos cuidados que evitan la mayoría de las diferencias
1. El valor es **exacto**: `"Entregado"` no es lo mismo que `"entregado"` ni que `"Entregado "` (con espacio al final). El perfil de la Parte 2 te ayudó a ver si hay espacios.
2. Si tu fuente tiene **filas repetidas**, tu conteo a mano puede incluirlas o no. Anota qué hiciste.

### TU TURNO · define y cuenta tus tres conteos

**Qué hace la celda:** define `CONTEOS`. Cada ficha tiene `"a_mano"` en `None`. **Reemplázalo por el número que contaste tú**, antes de mirar el resultado del código.

Los ejemplos son de la muestra sintética. Cambia `columna` y `valor` por los de tu fuente.

In [ ]:
CONTEOS = [
    {"nombre": "Filas de la tabla",        "tipo": "filas",                                              "a_mano": None},
    {"nombre": "Vacíos en 'estado'",       "tipo": "vacios",    "columna": "estado",                     "a_mano": None},
    {"nombre": "Pedidos 'Entregado'",      "tipo": "categoria", "columna": "estado", "valor": "Entregado", "a_mano": None},
]

### Calcular y comparar

**Qué hace la celda:**
- `calcular_conteo` hace el conteo de una ficha sobre una tabla.
- `verificar_conteos` calcula los tres, los compara con tu número y devuelve una tabla con la columna **coincide**.

**Cómo leer el resultado:**
- **Coincide:** tu conteo y el del código dicen lo mismo.
- **No coincide:** no cambies tu número para que cuadre. Busca la causa (espacios, mayúsculas, filtro olvidado, filas repetidas).
- **Sin comprobar:** todavía no escribiste tu conteo a mano.

Los conteos que **coinciden** son los que puedes llevar a tu informe.

In [ ]:
def calcular_conteo(tabla, ficha):
    tipo = ficha["tipo"]
    if tipo == "filas":
        return int(len(tabla))
    if tipo == "vacios":
        if ficha.get("columna"):
            return int(tabla[ficha["columna"]].isna().sum())
        return int(tabla.isna().sum().sum())
    if tipo == "categoria":
        return int((tabla[ficha["columna"]] == ficha["valor"]).sum())
    if tipo == "distintos":
        return int(tabla[ficha["columna"]].nunique())
    if tipo == "suma":
        return int(tabla[ficha["columna"]].sum())
    raise ValueError(f"Tipo de conteo desconocido: {tipo}")


def verificar_conteos(tabla, conteos):
    filas = []
    for ficha in conteos:
        codigo = calcular_conteo(tabla, ficha)
        mano = ficha.get("a_mano")
        if mano is None:
            estado = "Sin comprobar"
        else:
            estado = "Coincide" if int(mano) == codigo else "No coincide"
        filas.append({"conteo": ficha["nombre"], "a mano": mano, "código": codigo, "resultado": estado})
    return pd.DataFrame(filas)


resultado_conteos = verificar_conteos(df, CONTEOS)
resultado_conteos

## Parte 6 · Una pregunta que tu fuente permite y otra que no

Una fuente solo responde lo que tiene. Pedirle más a una IA es la forma más rápida de recibir una respuesta inventada.

### Cómo decidirlo
Para cada pregunta, escribe **qué columnas necesitaría**. Si todas existen y tienen datos suficientes, la fuente la permite. Si falta alguna, no.

### TU TURNO
**Qué hace la celda:** guarda tus dos preguntas y las columnas que necesita cada una. `puede_responder` revisa si esas columnas existen y cuántos vacíos tienen.

**Un detalle importante:** que la columna exista no basta. Si el 40 % de sus celdas están vacías, tu respuesta se apoya en menos de la mitad de los datos, y tienes que decirlo.

In [ ]:
# TU TURNO: reemplaza con preguntas de TU fuente. Estos son ejemplos de la muestra sintética.
PREGUNTA_SI = "¿Cuántos pedidos hay por estado?"
COLUMNAS_SI = ["estado"]

PREGUNTA_NO = "¿Por qué se cancelan los pedidos?"
COLUMNAS_NO = ["motivo_cancelacion"]


def puede_responder(tabla, columnas):
    faltan = [c for c in columnas if c not in tabla.columns]
    if faltan:
        return False, f"faltan las columnas {faltan}"
    detalle = ", ".join(f"{c} tiene {100 * tabla[c].isna().mean():.0f}% de vacíos" for c in columnas)
    return True, detalle


for etiqueta, pregunta, columnas in [("Permitida", PREGUNTA_SI, COLUMNAS_SI), ("No permitida", PREGUNTA_NO, COLUMNAS_NO)]:
    ok, detalle = puede_responder(df, columnas)
    print(f"{etiqueta}: {pregunta}")
    print("  ->", "La fuente puede responder" if ok else "La fuente NO puede responder", "|", detalle)
    print()

## Parte 7 · Registro y archivos para entregar

### El registro tiene cuatro datos
| # | Dato | Qué guarda esta plantilla |
|---|---|---|
| 1 | Fuente | `OPCION`, `URL` y el texto de origen |
| 2 | Fecha y hora | El momento de la consulta |
| 3 | Instrucción | Qué cargaste y las fichas de conteo (qué contaste, dónde, con qué valor) |
| 4 | Comprobación | Tus números a mano, los del código y si coincidieron |

### Qué hace la celda
Arma el registro como un diccionario de Python, lo guarda como **JSON** (para que el código pueda leerlo en la prueba en frío) y como **CSV** (para que una persona lo lea en una hoja). También guarda tu diccionario de datos.

**Qué esperar:** tres líneas que dicen qué archivos se guardaron. En Colab, además, se descargan a tu computador. **Descárgalos**: el espacio de Colab se borra al cerrar la sesión.

> Si tu tipo de fuente es `"copia"`, el registro lo dice. Eso es lo correcto: una copia **marcada** es un resultado honesto.

In [ ]:
registro = {
    "fuente": {"opcion": OPCION, "url": URL, "clave_lista": CLAVE_LISTA, "origen": ORIGEN, "tipo": TIPO_FUENTE},
    "momento": MOMENTO,
    "instruccion": {
        "cargar": f"cargar_fuente() con OPCION='{OPCION}'",
        "conteos": CONTEOS,
    },
    "comprobacion": [
        {"conteo": fila["conteo"], "a_mano": (None if pd.isna(fila["a mano"]) else int(fila["a mano"])),
         "codigo": int(fila["código"]), "resultado": fila["resultado"]}
        for _, fila in resultado_conteos.iterrows()
    ],
}

with open("registro_proyecto.json", "w", encoding="utf-8") as archivo:
    json.dump(registro, archivo, ensure_ascii=False, indent=2)

registro_csv = pd.DataFrame({
    "dato": ["1 · Fuente", "2 · Fecha y hora", "3 · Instrucción", "4 · Comprobación"],
    "valor": [
        ORIGEN,
        MOMENTO,
        registro["instruccion"]["cargar"] + " | conteos: " + "; ".join(c["nombre"] for c in CONTEOS),
        "; ".join(f"{c['conteo']}: {c['resultado']}" for c in registro["comprobacion"]),
    ],
})
registro_csv.to_csv("registro_proyecto.csv", index=False, encoding="utf-8-sig")
tabla_diccionario.to_csv("diccionario_proyecto.csv", index=False, encoding="utf-8-sig")

print("Guardé registro_proyecto.json")
print("Guardé registro_proyecto.csv")
print("Guardé diccionario_proyecto.csv")

try:
    from google.colab import files
    for nombre in ("registro_proyecto.json", "registro_proyecto.csv", "diccionario_proyecto.csv"):
        files.download(nombre)
except ImportError:
    print("No estoy en Colab: los archivos quedaron en la carpeta del notebook.")

## Parte 8 · Prueba en frío

La prueba en frío contesta una sola pregunta: **¿puedo repetir mi resultado usando solo mi registro?**

### Cómo hacerla bien
1. **Reinicia la sesión:** en Colab, menú **Entorno de ejecución → Reiniciar sesión**. Así se borran todas las variables.
2. **No vuelvas a ejecutar las partes anteriores.** Solo ejecuta la Parte 0 (las importaciones), las **funciones** de la Parte 1 y la Parte 5, y esta celda.
3. La función lee `registro_proyecto.json`, vuelve a cargar la fuente **como dice el registro**, repite tus tres conteos y compara con lo que habías anotado.
4. Anota qué cambió y por qué. Solo hay dos explicaciones posibles: **cambió la fuente** o **cambió la forma de preguntar**.

### Qué hace la función `prueba_en_frio`
- Lee el registro guardado.
- Carga la fuente otra vez (las fuentes `url`, `api` y `muestra` se pueden recargar; un archivo subido, no: tendrías que subirlo de nuevo).
- Calcula cada conteo y lo compara con el `codigo` que tenía el registro.
- Si el número es igual, dice «igual». Si es distinto, dice que **la fuente cambió** (porque la instrucción del registro es la misma).

**Qué esperar con una copia o la muestra:** todo «igual», porque no cambia. Con una fuente viva, puede aparecer una diferencia. Eso **no es un error**: es exactamente lo que le pasó a Camila.

In [ ]:
def prueba_en_frio(ruta_registro="registro_proyecto.json"):
    with open(ruta_registro, encoding="utf-8") as archivo:
        reg = json.load(archivo)

    opcion = reg["fuente"]["opcion"]
    if opcion == "url":
        tabla, _, _ = leer_url(reg["fuente"]["url"])
    elif opcion == "api":
        tabla, _, _ = leer_api(reg["fuente"]["url"], reg["fuente"]["clave_lista"])
    elif opcion == "muestra":
        tabla, _, _ = muestra_sintetica()
    else:
        print("Esta fuente es un archivo subido. Súbelo de nuevo con leer_archivo() y llama a")
        print("prueba_en_frio_con(tabla) pasando la tabla.")
        return None
    return prueba_en_frio_con(tabla, reg)


def prueba_en_frio_con(tabla, reg=None, ruta_registro="registro_proyecto.json"):
    if reg is None:
        with open(ruta_registro, encoding="utf-8") as archivo:
            reg = json.load(archivo)
    filas = []
    for ficha, anterior in zip(reg["instruccion"]["conteos"], reg["comprobacion"]):
        ahora = calcular_conteo(tabla, ficha)
        antes = anterior["codigo"]
        filas.append({
            "conteo": ficha["nombre"],
            "en el registro": antes,
            "ahora": ahora,
            "diagnóstico": "igual" if ahora == antes else "cambió la fuente",
        })
    print("Fuente del registro:", reg["fuente"]["origen"])
    print("Consulta original  :", reg["momento"])
    print("Prueba en frío     :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    return pd.DataFrame(filas)


prueba_en_frio()

### Opcional · simula que la fuente cambió

**Solo funciona con `OPCION = "muestra"`.** Esta celda agrega cinco pedidos nuevos a una copia en memoria y repite la prueba para que veas cómo se ve un cambio de fuente. No toca ningún archivo ni ninguna fuente real.

**Predice:** ¿qué conteos cambian y cuáles no?

In [ ]:
if OPCION == "muestra":
    nuevos = pd.DataFrame([
        {"pedido": f"N{n}", "fecha": "05/11/2026", "ciudad": "Medellín", "producto": "Café",
         "cantidad": 2, "valor_unitario": 18000, "estado": "Entregado"}
        for n in range(1, 6)
    ])
    tabla_cambiada = pd.concat([muestra_sintetica()[0], nuevos], ignore_index=True)
    display(prueba_en_frio_con(tabla_cambiada))
else:
    print("Esta simulación es solo para la muestra sintética.")

## Parte 9 · Una muestra segura para preguntar a una IA

Si vas a pegar datos en una herramienta con IA, pega **una muestra pequeña**, no toda la tabla. Así gastas menos de tu límite de uso, cuidas tus datos y la IA recibe menos para equivocarse.

### Qué hace la celda
- Si la Parte 3 dio alertas de datos personales, **se detiene** y no imprime nada. Para continuar tendrías que escribir `CONFIRMO_SIN_DATOS_PERSONALES = True`; hazlo solo si de verdad confirmaste permiso.
- Si todo está bien, imprime las primeras 20 filas en texto, listas para pegar, y un **prompt de ejemplo** que incluye los dos datos que la IA necesita para no inventar: **el diccionario** y **qué no sabes**.

### Después de pegar
Compara siempre lo que responda con **tus tres conteos**. Si no coincide, gana tu conteo comprobado, no la IA.

In [ ]:
CONFIRMO_SIN_DATOS_PERSONALES = False   # cámbialo a True solo si confirmaste permiso

if alertas and not CONFIRMO_SIN_DATOS_PERSONALES:
    print("Me detengo: la Parte 3 encontró columnas que parecen datos personales:", alertas)
    print("Quítalas o confirma permiso antes de pegar datos en una IA.")
else:
    muestra_csv = df.head(20).to_csv(index=False)
    glosario = "\n".join(
        f"- {c}: {d['significado']}. Formato: {d['formato']}. Vacío: {d['vacio_significa']}."
        for c, d in DICCIONARIO.items()
    )
    prompt = (
        "Estas son 20 filas de una fuente más grande. No inventes columnas ni valores.\n"
        "Si la pregunta no se puede responder con estas columnas, dilo.\n\n"
        f"Diccionario de datos:\n{glosario}\n\n"
        f"Datos (CSV):\n{muestra_csv}\n"
        f"Pregunta: {PREGUNTA_SI}"
    )
    print(prompt)

## Parte 10 · Si algo falla

| Lo que ves | Qué significa | Qué hacer |
|---|---|---|
| `HTTPError: 401` o `403` | La hoja no está publicada o es privada | Publícala como CSV (Parte 1) o usa otra opción |
| `HTTPError: 404` | El enlace está mal copiado | Cópialo de nuevo, completo, sin espacios al final |
| `URLError` o `ConnectionError` | No hay conexión o el sitio no responde | Revisa tu internet. Usa `"subir"` o `"muestra"` mientras tanto |
| `ParserError` al leer una URL | No es un CSV (suele ser la página de la hoja) | El enlace debe terminar en `output=csv` |
| `ValueError: La API devolvió un objeto…` | La lista está dentro de una clave | Escribe esa clave en `CLAVE_LISTA` |
| `JSONDecodeError` | La respuesta no es JSON | Abre la dirección en el navegador y mira qué devuelve |
| `KeyError: 'nombre'` | La columna no existe con ese nombre exacto | Compara con `df.columns`. Cuidado con mayúsculas, tildes y espacios |
| `NameError` | Una celda anterior no se ejecutó | Ejecuta de arriba hacia abajo, en orden |
| `UnicodeDecodeError` al subir | Codificación distinta | Prueba `pd.read_csv(ruta, encoding="latin-1")` |
| El conteo no coincide | Espacios, mayúsculas, filtro olvidado o filas repetidas | Mira `solo espacios` en el perfil y `repetidas` |
| Se acabó el límite de la IA | Plan gratuito con tope | Este notebook funciona sin IA. Anota en tu registro la herramienta y la hora del límite |

**Regla general:** lee el **último renglón** del error; casi siempre dice qué pasó.

### Si no puedes conectar tu fuente
Usa una **copia estática** y escribe en tu registro que es una copia y con qué fecha la sacaste. Una copia marcada es un resultado honesto; una copia que parece una fuente viva, no.

### Si no tienes datos
Usa `OPCION = "muestra"` y anota que son datos inventados y qué **no** puedes afirmar con ellos.

## Glosario

- **Fuente viva:** conexión a los datos originales; lo que lees puede cambiar.
- **Copia estática:** fotografía de los datos en un momento; hay que marcarla como copia.
- **DataFrame (`df`):** la tabla de Pandas.
- **Celda vacía (`NaN`):** celda sin dato; su significado lo decides tú.
- **Diccionario de datos:** qué significa cada columna y cada vacío.
- **Registro de la consulta:** fuente, fecha y hora, instrucción y comprobación.
- **Ficha de conteo:** una pequeña descripción de qué se cuenta, en qué columna y con qué valor.
- **Prueba en frío:** repetir tu resultado usando solo el registro.
- **Solo lectura:** acceso que permite leer y no cambiar.
- **API:** dirección de internet que responde con datos.

## Lista final: ¿terminé?
- [ ] Mi fuente está cargada y sé si es **viva** o **copia**.
- [ ] Mi perfil de columnas está revisado: tipos, vacíos, espacios y repetidos.
- [ ] Revisé que no haya datos personales.
- [ ] Mi diccionario no tiene avisos.
- [ ] Mis tres conteos están comprobados a mano y los que no coinciden tienen una causa anotada.
- [ ] Tengo una pregunta que mi fuente permite y otra que no.
- [ ] Descargué el registro y el diccionario.
- [ ] Hice la prueba en frío.

**Para el ticket de salida:** escribe en el chat qué representa **una fila** de tu fuente y **un conteo que comprobaste**. Sin datos personales ni de clientes.

**Próximo encuentro:** limpiamos y transformamos tus datos. Trae marcados dos problemas de tu fuente: vacíos, repetidos o valores raros. Hoy no borres nada.